# Time Series Forecasting — LSTM Seq2Seq (v2)
**Forecast:** 72 hours ahead (288 steps @ 15-min intervals)

### Fixes applied vs v1:
- Seq2Seq LSTM decoder (replaces flat `Dense(288)`) — fixes oscillations
- Huber loss (replaces MSE) — robust to spikes, fixes bias
- Bottleneck `16 → 64` — more capacity for 3-day task
- `time_steps` `672 → 1344` (2 weeks of context)
- `stride` `288 → 96` — more overlapping windows, more training samples
- `ReduceLROnPlateau` + `EarlyStopping` callbacks
- Wider LSTM layers: `64/32 → 128/64`

## 1. Imports

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
import requests
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

## 2. Config & Hyperparameters

In [ ]:
# ── Database connection ───────────────────────────────────────────────────────
VIRTUOSO_URL = "http://localhost:8890/sparql"
GRAPH_URI    = "http://example.com/Gent-Terneuzen"
USERNAME     = "dba"
PASSWORD     = "dba"
AUTH         = (USERNAME, PASSWORD)

# ── Hyperparameters ───────────────────────────────────────────────────────────
TIME_STEPS = 1344   # 2 weeks @ 15-min = 2 * 7 * 24 * 4  (was 672 / 1 week)
N_FUTURE   = 288    # 3 days  @ 15-min = 3 * 24 * 4       (was 24 / 6 hours)
STRIDE     = 96     # overlapping windows → more training samples (was 288)
BATCH_SIZE = 32

SENSORS    = ['289435042', '289423042', '289429042', '289441042']

## 3. Fetch Data from Virtuoso

In [ ]:
# ── Identify unique sensors ───────────────────────────────────────────────────
sensor_set = set()

sensor_query = f"""
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    SELECT DISTINCT ?sensor
    WHERE {{
        GRAPH <{GRAPH_URI}> {{
            ?obs a sosa:Observation ;
                 sosa:madeBySensor ?sensor .
        }}
    }}
    """

res = requests.get(
    VIRTUOSO_URL,
    params={'query': sensor_query, 'format': 'application/sparql-results+json'}
)

if res.status_code != 200:
    print(f"Error: {res.status_code} — {res.text}")
else:
    print("Sensors identified successfully!")

for row in res.json()['results']['bindings']:
    sensor_set.add(row['sensor']['value'])

print(f"Found {len(sensor_set)} sensors: {sensor_set}")

In [ ]:
# ── Fetch and pivot all sensor data ──────────────────────────────────────────
final_df = pd.DataFrame()

print("Fetching sensor data...")

for sensor_uri in sensor_set:
    column_name = sensor_uri.split('/')[-1]

    query = f"""
        PREFIX sosa: <http://www.w3.org/ns/sosa/>
        PREFIX ex:   <http://example.com/attributes/>
        SELECT ?time ?value ?unixtime
        WHERE {{
            GRAPH <{GRAPH_URI}> {{
                ?obs a sosa:Observation ;
                    sosa:resultTime ?time ;
                    sosa:hasSimpleResult ?value ;
                    ex:unixTimestamp ?unixtime ;
                    sosa:madeBySensor <{sensor_uri}> .
            }}
        }}
    """

    res = requests.get(
        VIRTUOSO_URL,
        params={'query': query, 'format': 'application/sparql-results+json'}
    )

    if res.status_code == 200:
        bindings = res.json()['results']['bindings']
        temp_data = [
            {
                'time':      row['time']['value'],
                column_name: float(row['value']['value']),
                'unixtime':  int(row['unixtime']['value'])
            }
            for row in bindings
        ]
        temp_df = pd.DataFrame(temp_data)

        if not temp_df.empty:
            temp_df['time'] = pd.to_datetime(temp_df['time'])
            if final_df.empty:
                final_df = temp_df
            else:
                final_df = pd.merge(final_df, temp_df, on=['time', 'unixtime'], how='outer')
            print(f"  Added sensor: {column_name}")

final_df = final_df.sort_values('time').set_index('time')
print(f"\nFinal shape: {final_df.shape}")
final_df.head()

## 4. Preprocessing

In [ ]:
# ── Plot raw sensor data ──────────────────────────────────────────────────────
df_clean = final_df.copy()
df_clean = df_clean.ffill().bfill()
df_clean['average_conductivity'] = df_clean[SENSORS].mean(axis=1)

fig, axes = plt.subplots(nrows=5, ncols=1, figsize=(14, 12), sharex=True)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
sensors_with_avg = SENSORS + ['average_conductivity']

for i, sensor in enumerate(sensors_with_avg):
    axes[i].plot(df_clean.index, df_clean[sensor], color=colors[i], linewidth=0.8)
    axes[i].set_ylabel("Value")
    axes[i].legend([f"Sensor {sensor}"], loc='upper right')
    axes[i].grid(True, alpha=0.3)

plt.suptitle("Sensor Data", fontsize=14)
plt.xlabel("Time")
plt.xticks(rotation=45)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# ── Train / test split → scale ────────────────────────────────────────────────
# Split FIRST, then scale — prevents data leakage
series = df_clean['average_conductivity'].values.reshape(-1, 1)  # (n, 1)

split_index  = int(len(series) * 0.8)
train_data   = series[:split_index]
test_data    = series[split_index:]

scaler       = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_data)   # fit only on train
test_scaled  = scaler.transform(test_data)

print(f"Train samples: {len(train_scaled):,}  |  Test samples: {len(test_scaled):,}")

## 5. Sequence Creation

In [ ]:
def create_sequences(dataset, time_steps, n_future, stride):
    """
    dataset    : (n_samples, 1) scaled array
    Returns
      X : (samples, time_steps, 1)  — input sequences
      y : (samples, n_future)        — forecast horizon
    """
    X, y = [], []
    for i in range(0, len(dataset) - time_steps - n_future + 1, stride):
        X.append(dataset[i : i + time_steps, :])
        y.append(dataset[i + time_steps : i + time_steps + n_future, 0])
    return np.array(X), np.array(y)


X_train, y_train = create_sequences(train_scaled, TIME_STEPS, N_FUTURE, STRIDE)
X_test,  y_test  = create_sequences(test_scaled,  TIME_STEPS, N_FUTURE, STRIDE)

print(f"X_train: {X_train.shape}  →  (samples, time_steps, 1)")
print(f"y_train: {y_train.shape}  →  (samples, n_future)")
print(f"X_test:  {X_test.shape}")
print(f"y_test:  {y_test.shape}")

## 6. Model

### Architecture overview

**Stage 1** — LSTM Autoencoder pre-training (reconstruction task)  
Forces the encoder to learn a compact, meaningful representation of the time series before any forecasting.

**Stage 2** — Freeze encoder, train Seq2Seq forecast head  
The decoder LSTM unrolls step-by-step across the 288-step horizon, giving the model temporal structure in the output. This replaces the flat `Dense(n_future)` from v1 which caused oscillations.

**Stage 3** — Unfreeze all weights, fine-tune end-to-end at low LR

In [ ]:
def build_autoencoder(time_steps, n_features=1):
    """
    Bottleneck: 64-dim (was 16)
    Loss: Huber (was MSE)
    """
    inputs = layers.Input(shape=(time_steps, n_features), name='encoder_input')

    # Encoder
    x = layers.LSTM(128, return_sequences=True)(inputs)
    x = layers.Dropout(0.1)(x)
    x = layers.LSTM(64, return_sequences=False)(x)
    bottleneck = layers.Dense(64, activation='relu', name='bottleneck')(x)

    # Decoder
    x = layers.RepeatVector(time_steps)(bottleneck)
    x = layers.LSTM(64, return_sequences=True)(x)
    x = layers.Dropout(0.1)(x)
    x = layers.LSTM(128, return_sequences=True)(x)
    outputs = layers.TimeDistributed(layers.Dense(n_features), name='reconstruction')(x)

    autoencoder = Model(inputs, outputs, name='lstm_autoencoder')
    autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='huber'
    )
    return autoencoder


def build_seq2seq_forecaster(encoder, time_steps, n_future, n_features=1):
    """
    Seq2Seq decoder: LSTM unrolls across n_future steps.
    Replaces flat Dense(n_future) from v1.
    """
    forecast_input = layers.Input(shape=(time_steps, n_features), name='forecast_input')

    # Frozen encoder → bottleneck (batch, 64)
    z = encoder(forecast_input)

    # Decoder LSTM
    x = layers.RepeatVector(n_future)(z)               # (batch, n_future, 64)
    x = layers.LSTM(128, return_sequences=True)(x)
    x = layers.Dropout(0.1)(x)
    x = layers.LSTM(64, return_sequences=True)(x)
    x = layers.TimeDistributed(layers.Dense(1))(x)     # (batch, n_future, 1)
    outputs = layers.Reshape((n_future,))(x)           # (batch, n_future)

    forecaster = Model(forecast_input, outputs, name='seq2seq_forecaster')
    forecaster.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='huber'
    )
    return forecaster


def get_callbacks(monitor='val_loss'):
    return [
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor=monitor, factor=0.5, patience=4,
            min_lr=1e-7, verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor=monitor, patience=8,
            restore_best_weights=True, verbose=1
        )
    ]

### Stage 1 — Autoencoder pre-training

In [ ]:
autoencoder = build_autoencoder(time_steps=TIME_STEPS)
autoencoder.summary()

autoencoder.fit(
    X_train, X_train,
    epochs=40,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=get_callbacks(),
    verbose=1
)

### Stage 2 — Forecast head (encoder frozen)

In [ ]:
# Extract encoder and freeze its weights
encoder = Model(
    inputs  = autoencoder.input,
    outputs = autoencoder.get_layer('bottleneck').output,
    name    = 'encoder'
)
for layer in encoder.layers:
    layer.trainable = False

forecaster = build_seq2seq_forecaster(encoder, TIME_STEPS, N_FUTURE)
forecaster.summary()

forecaster.fit(
    X_train, y_train,
    epochs=40,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=get_callbacks(),
    verbose=1
)

### Stage 3 — End-to-end fine-tuning

In [ ]:
# Unfreeze all layers
for layer in encoder.layers:
    layer.trainable = True

# Recompile with low learning rate
forecaster.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='huber'
)

forecaster.fit(
    X_train, y_train,
    epochs=30,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=get_callbacks(),
    verbose=1
)

## 7. Evaluation

In [ ]:
# ── Per-step MAE and RMSE ─────────────────────────────────────────────────────
test_preds_scaled = forecaster.predict(X_test)   # (samples, N_FUTURE)

maes, rmses = [], []
print(f"{'Step':<8} {'MAE':>8} {'RMSE':>8}")
print("-" * 26)

for step in range(N_FUTURE):
    preds_real = scaler.inverse_transform(
        test_preds_scaled[:, step].reshape(-1, 1)
    ).flatten()
    true_real = scaler.inverse_transform(
        y_test[:, step].reshape(-1, 1)
    ).flatten()

    mae  = mean_absolute_error(true_real, preds_real)
    rmse = np.sqrt(mean_squared_error(true_real, preds_real))
    maes.append(mae)
    rmses.append(rmse)

    if step % 96 == 0 or step == N_FUTURE - 1:  # print every 24 h + last
        print(f"  t+{step+1:<4}  {mae:>7.4f}  {rmse:>7.4f}")

print(f"\nOverall MAE:  {np.mean(maes):.4f}")
print(f"Overall RMSE: {np.mean(rmses):.4f}")

In [ ]:
# ── MAE over forecast horizon ─────────────────────────────────────────────────
plt.figure(figsize=(12, 4))
plt.plot(range(1, N_FUTURE + 1), maes, color='steelblue', linewidth=1.5)
plt.axvline(96,  color='grey', linestyle='--', alpha=0.6, label='24 h')
plt.axvline(192, color='grey', linestyle='--', alpha=0.6, label='48 h')
plt.axvline(288, color='grey', linestyle='--', alpha=0.6, label='72 h')
plt.title('MAE vs Forecast Horizon')
plt.xlabel('Steps ahead (15-min intervals)')
plt.ylabel('MAE')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Last test window: predicted vs actual ─────────────────────────────────────
last_pred_scaled = forecaster.predict(X_test[-1:])

pred_real = scaler.inverse_transform(
    last_pred_scaled[0].reshape(-1, 1)
).flatten()
true_real = scaler.inverse_transform(
    y_test[-1].reshape(-1, 1)
).flatten()

steps = range(1, N_FUTURE + 1)
plt.figure(figsize=(16, 5))
plt.plot(steps, true_real,  label='Actual',    color='steelblue', linewidth=1.5)
plt.plot(steps, pred_real,  label='Predicted', color='tomato',    linewidth=1.5, linestyle='--')
plt.axvline(96,  color='grey', linestyle=':', alpha=0.6, label='24 h mark')
plt.axvline(192, color='grey', linestyle=':', alpha=0.6, label='48 h mark')
plt.title('Average Conductivity — 72 h forecast vs actual (LSTM Seq2Seq v2)')
plt.xlabel('Steps ahead (15-min intervals)')
plt.ylabel('Conductivity')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Future Forecast (beyond dataset)

In [ ]:
full_scaled   = np.concatenate([train_scaled, test_scaled], axis=0)
last_sequence = full_scaled[-TIME_STEPS:].reshape(1, TIME_STEPS, 1)

scaled_forecast = forecaster.predict(last_sequence)
forecast_real   = scaler.inverse_transform(
    scaled_forecast[0].reshape(-1, 1)
).flatten()

# ── Plot the future forecast ──────────────────────────────────────────────────
steps = range(1, N_FUTURE + 1)
plt.figure(figsize=(16, 5))
plt.plot(steps, forecast_real, color='tomato', linewidth=1.5)
plt.axvline(96,  color='grey', linestyle=':', alpha=0.6, label='24 h')
plt.axvline(192, color='grey', linestyle=':', alpha=0.6, label='48 h')
plt.axvline(288, color='grey', linestyle=':', alpha=0.6, label='72 h')
plt.title('Average Conductivity — 72 h future forecast')
plt.xlabel('Steps ahead (15-min intervals)')
plt.ylabel('Conductivity')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Forecast for next {N_FUTURE} steps (72 hours):")
for i, val in enumerate(forecast_real, 1):
    if i == 1 or i % 96 == 0:
        label = f"Day {i // 96}" if i % 96 == 0 else "t+1"
        print(f"  {label} (t+{i:3d}): {val:.4f}")

## 9. Save Models & Scaler

In [ ]:
forecaster.save("forecaster_v2.keras")
autoencoder.save("autoencoder_v2.keras")
encoder.save("encoder_v2.keras")
joblib.dump(scaler, "scaler_v2.pkl")

print("Saved:")
print("  forecaster_v2.keras  — use this for inference")
print("  autoencoder_v2.keras")
print("  encoder_v2.keras     — reuse to skip Stage 1 if retraining")
print("  scaler_v2.pkl        — always load alongside the model")